# EX_05 — Vector stores y retrieval (ejercicios)

**Notebook de referencia:** `notebook/05_Vectorstores_Retrieval.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Chunking

Implementa un chunker trivial por **número de caracteres** con solapamiento (`chunk_size`, `chunk_overlap`). Aplícalo a un texto largo en una lista de strings.


In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    chunks = []
    # Calculamos el avance real en cada iteración para respetar el solapamiento
    step = chunk_size - overlap 
    
    for i in range(0, len(text), step):
        # Extraemos la porción de texto desde 'i' hasta 'i + tamaño_del_chunk'
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)
        
    return chunks

# Prueba del código
long_text = "word " * 500  # Genera un texto de 2500 caracteres
resultados = chunk_text(long_text)

print(f"Número total de chunks generados: {len(resultados)}")
print(f"Longitud del primer chunk: {len(resultados[0])}")


## Actividad 2 — Embeddings + FAISS

Embedde los chunks (puede ser `sentence_transformers`) y construye un índice `faiss.IndexFlatIP` o `IndexFlatL2`. Recupera los top-3 para una query.

*Hint:* L2-normalize vectors if you treat inner product as cosine similarity.


In [1]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Definimos los "chunks" (datos de ejemplo) y cargamos el modelo
chunks = [
    "FAISS es una librería de Meta para búsqueda de similitud eficiente.",
    "Los embeddings convierten el texto en vectores numéricos densos.",
    "Python es un lenguaje de programación muy usado en IA.",
    "La similitud del coseno mide el ángulo entre dos vectores espaciales."
]
modelo = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Generar Embeddings y Normalizar (Aplicando el Hint de la imagen)
# "Hint: L2-normalize vectors if you treat inner product as cosine similarity."
embeddings = modelo.encode(chunks) 
faiss.normalize_L2(embeddings) 

# 3. Construir el Índice FAISS
dimension = embeddings.shape[1]      
# Usamos IndexFlatIP como sugiere la imagen
index = faiss.IndexFlatIP(dimension) 
index.add(embeddings)                

# 4. Realizar la Búsqueda (Query) para los "top-3"
query = "¿Qué tecnología sirve para buscar vectores?"
query_vector = modelo.encode([query])
faiss.normalize_L2(query_vector)     # Normalizar la query es vital aquí

k = 3 # Recupera los top-3 para una query
distancias, indices = index.search(query_vector, k)

# Imprimir resultados
print(f"Buscando: '{query}'\n")
for i in range(k):
    idx = indices[0][i]
    score = distancias[0][i]
    print(f"Top {i+1} [Similitud: {score:.4f}]: {chunks[idx]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Buscando: '¿Qué tecnología sirve para buscar vectores?'

Top 1 [Similitud: 0.5601]: La similitud del coseno mide el ángulo entre dos vectores espaciales.
Top 2 [Similitud: 0.5022]: Los embeddings convierten el texto en vectores numéricos densos.
Top 3 [Similitud: 0.4825]: FAISS es una librería de Meta para búsqueda de similitud eficiente.


## Actividad 3 — Métrica manual

Para una query y tres documentos **artificiales** (uno relevante, dos ruido), muestra scores de similitud y verifica que el relevante queda primero.


In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Definimos la query y 3 documentos artificiales (sintéticos)
query = "¿Cómo se hace una tortilla de patatas?"

doc_relevante = "Para preparar una buena tortilla necesitas freír patatas, batir huevos, mezclarlos y cuajar todo en la sartén."
doc_ruido_1 = "La Revolución Francesa fue un conflicto social y político que inició en 1789."
doc_ruido_2 = "Los agujeros negros son regiones del espacio con una gravedad tan fuerte que ni la luz puede escapar."

documentos = [doc_ruido_1, doc_relevante, doc_ruido_2] # Metemos el relevante en medio

# 2. Cargamos el modelo y generamos los embeddings (vectores)
modelo = SentenceTransformer('all-MiniLM-L6-v2')
query_vector = modelo.encode(query)
docs_vectores = modelo.encode(documentos)

# 3. Métrica manual: Cálculo de la Similitud del Coseno
# Fórmula: (A · B) / (||A|| * ||B||)
def calcular_similitud_coseno(v1, v2):
    producto_punto = np.dot(v1, v2)
    magnitud_v1 = np.linalg.norm(v1)
    magnitud_v2 = np.linalg.norm(v2)
    return producto_punto / (magnitud_v1 * magnitud_v2)

# Calculamos los scores para cada documento
scores = [calcular_similitud_coseno(query_vector, doc_vec) for doc_vec in docs_vectores]

# 4. Ordenar resultados y verificar
resultados_ordenados = sorted(zip(scores, documentos), reverse=True)

print(f"🔍 Query: '{query}'\n")
for i, (score, doc) in enumerate(resultados_ordenados):
    print(f"Top {i+1} [Score: {score:.4f}] -> {doc}")

# Verificación (Assert): Comprobamos programáticamente que el relevante es el número 1
assert resultados_ordenados[0][1] == doc_relevante, "¡Error! El documento relevante no quedó primero."
print("\n✅ Verificación superada: El documento relevante encabeza el ranking.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔍 Query: '¿Cómo se hace una tortilla de patatas?'

Top 1 [Score: 0.7499] -> Para preparar una buena tortilla necesitas freír patatas, batir huevos, mezclarlos y cuajar todo en la sartén.
Top 2 [Score: 0.4313] -> La Revolución Francesa fue un conflicto social y político que inició en 1789.
Top 3 [Score: 0.4090] -> Los agujeros negros son regiones del espacio con una gravedad tan fuerte que ni la luz puede escapar.

✅ Verificación superada: El documento relevante encabeza el ranking.
